<a href="https://colab.research.google.com/github/linhkid/pallas-forge/blob/main/notebooks/01_tiled_matmul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Tiled Matrix Multiplication with Pallas

The "hello world" of Pallas kernels. This notebook walks through:

1. Writing a tiled matmul kernel from scratch
2. Understanding `BlockSpec`, `grid`, and the TPU memory model
3. Testing correctness against JAX's built-in `jnp.matmul`
4. Exploring how block sizes affect the result

> **Works on both CPU and TPU.** On CPU, kernels run via Pallas interpret mode.
> On TPU (e.g. Colab), you'll see real performance differences between block sizes.

In [1]:
# ============================================================
# Colab / TPU Setup — run this cell first!
# ============================================================
# On Colab: Runtime > Change runtime type > TPU
# Skip this cell if running locally with pallas-forge already installed.
# ============================================================

import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "jax[tpu]", "-f",
            "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        ]
    )
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pallas-forge[viz] @ git+https://github.com/linhkid/pallas-forge.git@v0.1.1",
        ]
    )
    print("Colab setup complete!")
else:
    print("Running locally — skipping Colab install.")

Colab setup complete!


In [2]:
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX version: 0.7.2
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Backend: tpu


## 1. The simplest matmul — JAX baseline

Before writing any Pallas code, let's establish the baseline with JAX's built-in matmul.

In [3]:
# Create random matrices
key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)

M, K, N = 256, 256, 256
x = jax.random.normal(k1, (M, K), dtype=jnp.float32)
w = jax.random.normal(k2, (K, N), dtype=jnp.float32)

# JAX baseline
expected = jnp.matmul(x, w)
print(f"Input shapes: x={x.shape}, w={w.shape}")
print(f"Output shape: {expected.shape}")
print(f"Output dtype: {expected.dtype}")

Input shapes: x=(256, 256), w=(256, 256)
Output shape: (256, 256)
Output dtype: float32


## 2. Pallas tiled matmul

Now let's use the `tiled_matmul` kernel from pallas-forge. The key concept:

- The output matrix is divided into **tiles** of size `(block_m, block_n)`
- Each tile is computed by one kernel instance on the **grid**
- The K dimension is reduced by looping over chunks of size `block_k`
- `BlockSpec` tells Pallas how to slice the input arrays for each grid cell

In [4]:
from pallas_forge import tiled_matmul

# Default block sizes
result = tiled_matmul(x, w, block_m=128, block_k=128, block_n=128)

# Check correctness
max_error = jnp.max(jnp.abs(result - expected))
print(f"Max absolute error vs jnp.matmul: {max_error:.2e}")
print(f"Results match: {jnp.allclose(result, expected, atol=1e-4)}")

Max absolute error vs jnp.matmul: 0.00e+00
Results match: True


## 3. Exploring block sizes

The block size is the primary tuning knob. Let's see how different values
all produce correct results (on TPU, they would have very different performance).

In [5]:
# On TPU, block_k and block_n must be multiples of 128 (MXU lane dimension).
# block_m can be a multiple of 8 — but in practice you usually want >= 128 too.
block_configs = [
    (128, 128, 128),
    (256, 128, 128),
    (128, 256, 128),
    (128, 128, 256),
]

print(
    f"{'block_m':>8} {'block_k':>8} {'block_n':>8} {'grid_m':>8} {'grid_n':>8} {'max_err':>12} {'correct':>8}"
)
print("-" * 72)

for bm, bk, bn in block_configs:
    result = tiled_matmul(x, w, block_m=bm, block_k=bk, block_n=bn)
    max_err = float(jnp.max(jnp.abs(result - expected)))
    correct = bool(jnp.allclose(result, expected, atol=1e-4))
    grid_m = (M + bm - 1) // bm
    grid_n = (N + bn - 1) // bn
    print(f"{bm:>8} {bk:>8} {bn:>8} {grid_m:>8} {grid_n:>8} {max_err:>12.2e} {str(correct):>8}")

 block_m  block_k  block_n   grid_m   grid_n      max_err  correct
------------------------------------------------------------------------
     128      128      128        2        2     0.00e+00     True
     256      128      128        1        2     0.00e+00     True
     128      256      128        2        2     0.00e+00     True
     128      128      256        2        1     0.00e+00     True


## 4. Non-aligned dimensions

Real-world matrices rarely align perfectly to block boundaries.
The kernel handles this via automatic padding and unpadding.

In [6]:
# Intentionally non-aligned sizes — the kernel pads up internally.
M2, K2, N2 = 100, 200, 150
x2 = jax.random.normal(k1, (M2, K2), dtype=jnp.float32)
w2 = jax.random.normal(k2, (K2, N2), dtype=jnp.float32)

# Use TPU-valid block sizes (>= 128 for block_k and block_n).
result2 = tiled_matmul(x2, w2, block_m=128, block_k=128, block_n=128)
expected2 = jnp.matmul(x2, w2)

print(f"Input: ({M2}, {K2}) x ({K2}, {N2}) with block=128")
print(f"Output shape: {result2.shape} (correctly unpadded)")
print(f"Max error: {float(jnp.max(jnp.abs(result2 - expected2))):.2e}")
print(f"Correct: {bool(jnp.allclose(result2, expected2, atol=1e-4))}")

Input: (100, 200) x (200, 150) with block=128
Output shape: (100, 150) (correctly unpadded)
Max error: 0.00e+00
Correct: True


## 5. bfloat16 — the TPU-native dtype

TPU's MXU is optimized for bfloat16 matmuls. The kernel accumulates
in float32 internally for numerical stability, then casts back.

In [7]:
x_bf16 = x.astype(jnp.bfloat16)
w_bf16 = w.astype(jnp.bfloat16)

result_bf16 = tiled_matmul(x_bf16, w_bf16, block_m=128, block_k=128, block_n=128)
expected_bf16 = jnp.matmul(x_bf16, w_bf16)

print(f"Output dtype: {result_bf16.dtype}")
print(
    f"Max error: {float(jnp.max(jnp.abs(result_bf16.astype(jnp.float32) - expected_bf16.astype(jnp.float32)))):.2e}"
)
print(f"Correct (atol=0.1 for bf16): {bool(jnp.allclose(result_bf16, expected_bf16, atol=0.1))}")

Output dtype: bfloat16
Max error: 0.00e+00
Correct (atol=0.1 for bf16): True


## Key takeaways

1. **`BlockSpec`** maps grid indices to array slices — it's how Pallas knows which tile each kernel instance should work on
2. **Block sizes** don't affect correctness, but on TPU they dramatically affect performance (3-5x swings)
3. **Padding/unpadding** handles non-aligned dimensions transparently
4. **float32 accumulation** is essential for bf16 inputs to avoid precision loss

Next: [02_fused_rmsnorm.ipynb](02_fused_rmsnorm.ipynb) — a memory-bound kernel that shows VPU vs MXU tradeoffs.